# Collect current Alpaca news for the project tickers

This notebook calls Alpaca's [`/v1beta1/news`](https://data.alpaca.markets/v1beta1/news) endpoint and loads recent articles for symbols listed in `documentation/alpaca_tickers.txt`. It reads a rolling 24-hour window, requests symbols in manageable batches, follows pagination, and displays a deduplicated table.

## Before running

- Install `requests` and `pandas` in the notebook environment if they are not already available.
- Provide `ALPACA_API_KEY` and `ALPACA_SECRET_KEY` with keyring.
- Run the cells from top to bottom. News availability depends on your Alpaca account and feed permissions; HTTP/API errors are raised rather than hidden.

## Load the ticker list and configure the request

The file is located by walking upward from the notebook's working directory, so this works when the notebook is launched from the repository root or `src/db/`.

In [1]:
from datetime import datetime, timedelta, timezone
#from getpass import getpass
import keyring
from pathlib import Path
import os
import time

import pandas as pd
import requests

ENDPOINT = "https://data.alpaca.markets/v1beta1/news"
PAGE_SIZE = 50
SYMBOLS_PER_REQUEST = 40
LOOKBACK = timedelta(hours=24)

api_key = os.getenv("ALPACA_API_KEY") or keyring.get_password(
    "api.alpaca.markets", "ALPACA_API_KEY")

secret_key = os.getenv("ALPACA_SECRET_KEY") or keyring.get_password(
    "api.alpaca.markets", "ALPACA_SECRET_KEY")


ticker_file = next(
    (
        parent / "documentation" / "alpaca_tickers.txt"
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "documentation" / "alpaca_tickers.txt").is_file()
    ),
    None,
)
if ticker_file is None:
    raise FileNotFoundError(
        "Could not find documentation/alpaca_tickers.txt from the notebook's working directory."
    )

symbols = list(
    dict.fromkeys(
        line.strip().upper()
        for line in ticker_file.read_text(encoding="utf-8").splitlines()
        if line.strip() and not line.lstrip().startswith("#")
    )
)
if not symbols:
    raise ValueError(f"No ticker symbols found in {ticker_file}")

#api_key = os.getenv("ALPACA_API_KEY") or getpass("Alpaca API key: ")
#secret_key = os.getenv("ALPACA_SECRET_KEY") or getpass("Alpaca secret key: ")
if not api_key or not secret_key:
    raise ValueError("Both Alpaca API credentials are required.")

headers = {
    "accept": "application/json",
    "APCA-API-KEY-ID": api_key,
    "APCA-API-SECRET-KEY": secret_key,
}
end_time = datetime.now(timezone.utc)
start_time = end_time - LOOKBACK
print(f"Loaded {len(symbols)} symbols from {ticker_file}")
print(f"News interval (UTC): {start_time:%Y-%m-%d %H:%M:%S} to {end_time:%Y-%m-%d %H:%M:%S}")

Loaded 514 symbols from /home/djmead/Documents/Projects/Financial/documentation/alpaca_tickers.txt
News interval (UTC): 2026-10-12 20:53:02 to 2026-10-13 20:53:02


## Fetch and paginate news

Each request asks for up to 50 articles and uses Alpaca's `next_page_token` to retrieve the rest. Small ticker batches keep request URLs manageable. A short pause between requests helps avoid sending a burst of calls. Articles that appear in more than one batch are merged by ID.

In [ ]:
def to_rfc3339(value):
    return value.isoformat(timespec="seconds").replace("+00:00", "Z")

def retrieve_articles():
    news_by_id = {}
    request_count = 0

    for offset in range(0, len(symbols), SYMBOLS_PER_REQUEST):
        symbol_batch = symbols[offset : offset + SYMBOLS_PER_REQUEST]
        page_token = None

        while True:
            params = {
                "symbols": ",".join(symbol_batch),
                "start": to_rfc3339(start_time),
                "end": to_rfc3339(end_time),
                "sort": "desc",
                "limit": PAGE_SIZE,
                "include_content": "false",
            }
            if page_token:
                params["page_token"] = page_token

            response = requests.get(ENDPOINT, headers=headers, params=params, timeout=30)
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict) or not isinstance(payload.get("news"), list):
                raise ValueError("Unexpected Alpaca news response: expected an object with a 'news' list.")

            request_count += 1
            for article in payload["news"]:
                article_id = article.get("id")
                if article_id is None:
                    raise ValueError("Alpaca returned a news article without an ID.")

                if article_id not in news_by_id:
                    news_by_id[article_id] = article
                else:
                    known_symbols = set(news_by_id[article_id].get("symbols") or [])
                    known_symbols.update(article.get("symbols") or [])
                    news_by_id[article_id]["symbols"] = sorted(known_symbols)

            page_token = payload.get("next_page_token")
            if not page_token:
                break
            time.sleep(0.7)

        if offset + SYMBOLS_PER_REQUEST < len(symbols):
            time.sleep(0.7)

    articles = list(news_by_id.values())
    print(f"Received {len(articles)} unique articles in {request_count} API requests.")
    return articles

articles = retrieve_articles()

Received 402 unique articles in 16 API requests.


## Review the results

The table includes headline, publication time, author, matched ticker symbols, source, summary, and article URL. `include_content=false` keeps full article bodies out of the response.

In [3]:
columns = ["id", "created_at", "headline", "author", "symbols", "source", "summary", "url"]
news_df = pd.DataFrame.from_records(articles).reindex(columns=columns)

if news_df.empty:
    print("No news articles were returned for these symbols and this time window.")
else:
    news_df = news_df.sort_values("created_at", ascending=False).reset_index(drop=True)
    display(news_df)

    ticker_counts = (
        news_df.explode("symbols")["symbols"]
        .value_counts()
        .rename_axis("symbol")
        .rename("article_count")
        .to_frame()
    )
    display(ticker_counts)

,id,created_at,headline,author,symbols,source,summary,url
0,62341492,2026-10-13T20:51:37Z,"Applied Materials Stock Sinks for 6th Day, but...",Piero Cingari,"[AMAT, ASML, AVGO, KLAC, LRCX, MU, SKHY, SSNLF]",benzinga,Applied Materials stock fell 7% on investor da...,https://www.benzinga.com/analyst-stock-ratings...
1,62341348,2026-10-13T20:46:04Z,DOJ Says Saudi National Arrested For Installin...,Benzinga Newsdesk,[MSFT],benzinga,,https://www.benzinga.com/news/26/10/62341348/d...
2,62341261,2026-10-13T20:42:52Z,Avcon Industries Receives FAA Supplemental Typ...,Benzinga Newsdesk,"[BUKS, HONA]",benzinga,,https://www.benzinga.com/news/26/10/62341261/a...
3,62341241,2026-10-13T20:41:42Z,U.S. Department Of The Treasury Announced Enfo...,Benzinga Newsdesk,[BAC],benzinga,,https://www.benzinga.com/news/26/10/62341241/u...
4,62340990,2026-10-13T20:36:47Z,Dan Ives Says AI Stock Selloff ‘Opportunity to...,Chris Katje,"[CRM, CRWD, META, MSFT, NOW, NVDA, PLTR]",benzinga,"In a recent interview, Dan Ives names seven st...",https://www.benzinga.com/trading-ideas/long-id...
...,...,...,...,...,...,...,...,...
397,62312280,2026-10-12T21:04:07Z,"UBS Maintains Buy on Delta Air Lines, Lowers P...",Benzinga Newsdesk,[DAL],benzinga,,https://www.benzinga.com/news/26/10/62312280/u...
398,62312210,2026-10-12T21:00:18Z,"Citigroup Maintains Buy on Wynn Resorts, Lower...",Benzinga Newsdesk,[WYNN],benzinga,,https://www.benzinga.com/news/26/10/62312210/c...
399,62312209,2026-10-12T21:00:16Z,$100 Invested In Hilton Worldwide Holdings 5 Y...,Benzinga Insights,[HLT],benzinga,Hilton Worldwide Holdings (NYSE:HLT) has outpe...,https://www.benzinga.com/news/26/10/62312209/1...
400,62312145,2026-10-12T20:57:08Z,Citigroup Maintains Neutral on MGM Resorts Int...,Benzinga Newsdesk,[MGM],benzinga,,https://www.benzinga.com/news/26/10/62312145/c...


,article_count
symbol,
META,27
JPM,24
MSFT,20
WFC,20
GS,19
...,...
CF,1
TRGP,1
APA,1


## Optional: save the collected data

Run this cell to write the table to a CSV in the notebook's current working directory.

In [4]:
output_path = Path("../../documentation/alpaca_news.csv")
news_df.to_csv(output_path, index=False)
print(f"Saved {len(news_df)} articles to {output_path.resolve()}")

Saved 402 articles to /home/djmead/Documents/Projects/Financial/documentation/alpaca_news.csv


In [1]:
news_df

NameError: name 'news_df' is not defined